# Phase 01 - Command line interface

Argparse CLI for the generated `group3-starvenus.py`. This notebook is exported last (`90_`), so it
may refer to anything defined in the earlier notebooks.

Cells whose first line is `#| export` are copied into the generated file; `tools/export_py.py`
appends the `if __name__ == "__main__": main()` guard. Exported cells must use only the standard
library, numpy, matplotlib, cv2, sklearn, skimage, torch and torchvision, and no notebook syntax
(`!`, `%`, `display(`).

Global flags (`--data-root`, `--device`, `--seed-offset`, `--fast`) are accepted both before and
after the subcommand.

To implement a command in a later phase: define its handler (`def cmd_xxx(args) -> int`) in the
phase's notebook, then replace the `not_implemented(...)` entry in `COMMANDS` below.

In [ ]:
#| export
import argparse
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

DEFAULT_DATA_ROOT = "Dhan-Shomadhan"
DEVICE_CHOICES = ("auto", "cuda", "mps", "cpu")
DEFAULT_SPLITS = (0, 1, 2, 3, 4)

EXIT_OK = 0
EXIT_FAILURE = 1
EXIT_NOT_IMPLEMENTED = 3

## Experiment registry and `choices.json` (T3.1)

`EXPERIMENTS` holds every row of the README matrix (section 5), keyed by the experiment ID that is used
verbatim in run folders, tables and CLI flags. Rows of later phases exist as entries too; running
them fails with a message that names the phase or the missing choice, never with a traceback.

Three things are decided by data and live in `_work/results/choices.json`, which is written by the
gates of phase 04 (and `choose-lr`). A registry entry leaves the field empty (`None`) and
`resolve_experiment` fills it at run time:

```json
{
  "input_mode": {"value": "rect",          "evidence": {"E0-rect": {"mean": 0.91, "std": 0.02}, "...": {}}},
  "backbone":   {"value": "convnext_tiny", "evidence": {"...": {}}},
  "lr": {
    "resnet50":      {"value": 0.0003, "evidence": {"experiment": "E0-rect", "split": 0,
                                                     "val_mixed_macro_f1": {"1e-4": 0.88, "3e-4": 0.91}}},
    "convnext_tiny": {"value": 0.0001, "evidence": {"...": "..."}},
    "swin_t":        {"value": 0.0001, "evidence": {"...": "..."}}
  },
  "recommended": {"value": "E3-A", "evidence": {}}
}
```

* `input_mode` - gate G0, the winner of E0 ("best of E0" in the matrix). One of `squash`, `pad`, `rect`.
* `backbone` - gate G1, the winner among ResNet50 / ConvNeXt-T / Swin-T ("best of E2").
* `lr` - learning rate **per backbone**, chosen by validation mixed macro-F1 from the split-0 sweep over
  {1e-4, 3e-4}. Written by `choose-lr --backbone NAME`. The learning rate of the winning backbone is
  `lr[backbone]`; there is no separate top-level lr.
* `recommended` - gate G2 (phase 06), the experiment ID that `predict` uses by default.

**Aliases (one training, several names).** `E1-U` is the winning E0 row. `E1-WW` is `E1-WF`: both train on white
and pick the checkpoint on white val, so they are the same model, and the `E1-WF` metrics already contain the white
test. `X-domain-WW` is the W->F row of the winning backbone (`E1-WF`, `E2-cnx-WF` or `E2-swin-WF`). Aliases resolve to
the target's config and run folder (`FIXED_ALIASES`, `BACKBONE_ALIASES`), so `run` skips them once the target is done and
`summarize` still shows an `E1-WW` row (white test from the target's runs). `X-domain-FF` has no counterpart for
ConvNeXt-T / Swin-T, so it is a real run.

Each entry may be written as `{"value": ..., "evidence": ...}` (what the gates write) or as the bare value
(`"input_mode": "rect"`, `"lr": {"resnet50": 0.0003}`), which is convenient by hand. A key that is missing,
or has an invalid value, gives a `ChoicesError` that says which gate or command writes it.

In [ ]:
#| export
import difflib

CHOICES_FILENAME = "choices.json"
SWEEP_LRS = (1e-4, 3e-4)  # T3.4: split-0 learning-rate sweep
# Experiment whose split-0 run is swept for each backbone (04: ResNet50 on rect, the others on their U row).
SWEEP_EXPERIMENT = {"resnet50": "E0-rect", "convnext_tiny": "E2-cnx-U", "swin_t": "E2-swin-U"}
# Used by `--fast` when choices.json does not exist yet, so that every experiment ID can be sanity-checked.
FAST_PLACEHOLDERS = {"input_mode": "rect", "backbone": "resnet50", "lr": 3e-4}
CHOICE_SOURCES = {
    "input_mode": 'the winning E0 input mode ("best of E0"), written by gate G0 (phase 04, Batch 1)',
    "backbone": 'the winning backbone ("best of E2"), written by gate G1 (phase 04, Batch 3)',
    "lr": "the learning rate of this backbone, written by `choose-lr --backbone NAME` after the split-0 sweep "
    "(`run --exp <ID> --sweep-lr`)",
}


def choices_path(data_root) -> Path:
    return results_dir(data_root) / CHOICES_FILENAME


@dataclass(frozen=True)
class Choices:
    """The contents of choices.json plus its path (for error messages)."""

    path: Path
    data: dict

    def raw(self, key: str, sub: str | None = None):
        entry = self.data.get(key)
        if sub is not None:
            entry = entry.get(sub) if isinstance(entry, dict) else None
        if isinstance(entry, dict) and "value" in entry:
            entry = entry["value"]
        return entry

    def get(self, key: str, sub: str | None = None):
        """The validated choice, or ChoicesError that says which step writes it."""
        value = self.raw(key, sub)
        label = key if sub is None else f"{key}[{sub}]"
        if value is None:
            where = f"{self.path} does not exist" if not self.path.is_file() else f"{self.path} has no such entry"
            raise ChoicesError(f"{where} (needed: {label}). It holds {CHOICE_SOURCES[key]}. Schema: see 90_cli.ipynb.")
        valid = {
            "input_mode": lambda v: v in INPUT_MODES,
            "backbone": lambda v: v in BACKBONES,
            "lr": lambda v: isinstance(v, (int, float)) and not isinstance(v, bool) and 0 < v < 1,
        }[key]
        if not valid(value):
            options = {"input_mode": INPUT_MODES, "backbone": BACKBONES, "lr": "a number in (0, 1)"}[key]
            raise ChoicesError(f"{self.path}: invalid {label} = {value!r} (expected {options})")
        return float(value) if key == "lr" else value


def load_choices(data_root) -> Choices:
    path = choices_path(data_root)
    if not path.is_file():
        return Choices(path, {})
    try:
        data = json.loads(read_file_bytes(path).decode("utf-8"))
    except ValueError as exc:
        raise ChoicesError(f"{path} is not valid JSON: {exc}") from exc
    if not isinstance(data, dict):
        raise ChoicesError(f"{path}: expected a JSON object at the top level")
    return Choices(path, data)


def write_choice(data_root, key: str, value, evidence=None, *, sub: str | None = None) -> Path:
    """Set choices[key] (or choices[key][sub]) to {"value": value, "evidence": evidence}, keeping the rest."""
    choices = load_choices(data_root)
    data = dict(choices.data)
    entry = {"value": value, "evidence": evidence if evidence is not None else {}}
    if sub is None:
        data[key] = entry
    else:
        group = dict(data[key]) if isinstance(data.get(key), dict) else {}
        group[sub] = entry
        data[key] = group
    write_json(choices.path, data, indent=2)
    return choices.path


def _row(exp_id: str, backbone, train, test_subsets, **kwargs) -> ExperimentConfig:
    train = tuple(train)
    return ExperimentConfig(
        exp_id=exp_id,
        backbone=backbone,
        train_domains=train,
        select_domains=train,
        test_subsets=tuple(test_subsets),
        input_mode=kwargs.pop("input_mode", None),
        **kwargs,
    )


_U, _W, _F = DOMAINS, ("white",), ("field",)
_ALL3 = ("white", "field", "mixed")
_WF_TEST = ("field", "white")  # field is the main result of a W->F row, white the reference

_ROWS = (
    # E0: input-size ablation, ResNet50 on white + field, equal pixel budget.
    _row("E0-squash", "resnet50", _U, _ALL3, input_mode="squash", note="320x320, aspect distorted"),
    _row("E0-pad", "resnet50", _U, _ALL3, input_mode="pad", note="320x320 letterbox"),
    _row("E0-rect", "resnet50", _U, _ALL3, input_mode="rect", note="448x224 (HxW)"),
    # E1: ResNet50 baseline and domain experiments at the winning E0 input mode.
    _row("E1-U", "resnet50", _U, _ALL3, alias_of_winner="E0", note="alias of the winning E0 row, no retraining"),
    _row("E1-WW", "resnet50", _W, ("white",), note="train white, test white"),
    _row("E1-FF", "resnet50", _F, ("field",), note="train field, test field"),
    _row("E1-WF", "resnet50", _W, _WF_TEST, note="train white, test field (cross-domain)"),
    # E2: ConvNeXt-T and Swin-T.
    _row("E2-cnx-U", "convnext_tiny", _U, _ALL3),
    _row("E2-cnx-WF", "convnext_tiny", _W, _WF_TEST),
    _row("E2-swin-U", "swin_t", _U, _ALL3),
    _row("E2-swin-WF", "swin_t", _W, _WF_TEST),
    # Later phases: runnable only once their hooks exist.
    _row("E3-A", None, _U, _ALL3, use_composites=True, note="Method A (phase 05): composites as extra training samples"),
    _row("E3-A-WF", None, _W, _WF_TEST, use_composites=True, note="white + composites; field images are only backgrounds"),
    _row("S-seg", "lraspp_mobilenet_v3_large", _U, (), kind="segmenter", note="leaf segmenter (phase 06)"),
    _row("E4-B", None, _U, _ALL3, seg_crop=True, note="Method B (phase 06): segment, crop, classify"),
    _row("X-domain-WW", None, _W, ("white",), note="best backbone, train white, test white"),
    _row("X-domain-FF", None, _F, ("field",), note="best backbone, train field, test field"),
    _row("X-gradcam", None, _U, (), kind="analysis", note="Grad-CAM shortcut analysis (phase 07), see the gradcam command"),
)
EXPERIMENTS: dict[str, ExperimentConfig] = {row.exp_id: row for row in _ROWS}
# `run --exp <GROUP>` runs every member in order.
EXPERIMENT_GROUPS = {
    "E0": ("E0-squash", "E0-pad", "E0-rect"),
    "E1": ("E1-U", "E1-WW", "E1-FF", "E1-WF"),
    "E2": ("E2-cnx-U", "E2-cnx-WF", "E2-swin-U", "E2-swin-WF"),
    "X-domain": ("X-domain-WW", "X-domain-FF"),
}
# Rows that are the same training as another row (same data, selection domains, backbone, input mode, lr and
# switches; they differ only in which test subset the report highlights). They have no run folder of their own:
# like E1-U they resolve to the target's config and folder, so nothing is trained twice. Aliases do not change
# any ExperimentConfig, so no existing fingerprint moves.
#   E1-WW: train white, select on white, same as E1-WF (whose metrics also hold the white test as reference).
FIXED_ALIASES = {"E1-WW": "E1-WF"}
#   X-domain-WW (best backbone, train white): the W->F row of the winning backbone is the same training.
#   X-domain-FF has no counterpart for ConvNeXt-T / Swin-T (there is no E2 F->F row), so it is not aliased.
BACKBONE_ALIASES = {"X-domain-WW": {"resnet50": "E1-WF", "convnext_tiny": "E2-cnx-WF", "swin_t": "E2-swin-WF"}}
NON_CLASSIFIER_MESSAGE = {
    "segmenter": "is the leaf segmenter, trained by the segmentation pipeline of phase 06 (06-segmentation-method-b.md)",
    "analysis": "is a Grad-CAM analysis, produced by the `gradcam` command of phase 07 (07-analysis.md)",
}


def run_folder_name(exp_id: str, choices: Choices) -> str:
    """Folder (under runs/) that holds the results of an experiment ID: itself, except for the E1-U alias."""
    if exp_id not in EXPERIMENTS:
        expand_experiment_ids(exp_id)  # raises with suggestions
    if exp_id in FIXED_ALIASES:
        return run_folder_name(FIXED_ALIASES[exp_id], choices)
    if exp_id in BACKBONE_ALIASES:
        return run_folder_name(BACKBONE_ALIASES[exp_id][choices.get("backbone")], choices)
    base = EXPERIMENTS[exp_id]
    return f"{base.alias_of_winner}-{choices.get('input_mode')}" if base.alias_of_winner else exp_id


def expand_experiment_ids(text: str) -> list[str]:
    """'E0' -> its three rows; 'E1-U,E1-WW' -> two rows. Unknown IDs raise UnknownExperimentError."""
    ids: list[str] = []
    for part in (p.strip() for p in text.split(",") if p.strip()):
        for exp_id in EXPERIMENT_GROUPS.get(part, (part,)):
            if exp_id not in EXPERIMENTS:
                close = difflib.get_close_matches(exp_id, [*EXPERIMENTS, *EXPERIMENT_GROUPS], n=3)
                hint = f" Did you mean: {', '.join(close)}?" if close else ""
                raise UnknownExperimentError(
                    f"unknown experiment {exp_id!r}.{hint} Known: {', '.join([*EXPERIMENTS, *EXPERIMENT_GROUPS])}"
                )
            if exp_id not in ids:
                ids.append(exp_id)
    if not ids:
        raise UnknownExperimentError("no experiment ID given")
    return ids


@dataclass(frozen=True)
class ResolvedExperiment:
    config: ExperimentConfig  # fully concrete: backbone, input_mode and lr are set
    requested: str  # the ID that was asked for (differs from config.exp_id for the E1-U alias)
    choices_used: dict  # which values came from choices.json (or FAST placeholders), for config.json
    notes: tuple[str, ...] = ()


def resolve_experiment(
    exp_id: str, choices: Choices, *, lr: float | None = None, placeholders: bool = False
) -> ResolvedExperiment:
    """Turn a registry entry into a concrete config.

    ``lr`` overrides the choices.json learning rate (sweeps, ``--lr``). ``placeholders=True`` (``--fast``)
    substitutes FAST_PLACEHOLDERS for choices that do not exist yet, and says so in ``notes``.
    """
    if exp_id not in EXPERIMENTS:
        expand_experiment_ids(exp_id)  # raises with a helpful message
    base = EXPERIMENTS[exp_id]
    if base.kind != "classifier":
        raise NotReadyError(f"{exp_id} {NON_CLASSIFIER_MESSAGE[base.kind]}; it cannot be run as a classifier")
    used: dict = {}
    notes: list[str] = []

    def pick(key: str, sub: str | None = None):
        label = key if sub is None else f"{key}[{sub}]"
        try:
            value = choices.get(key, sub)
            used[label] = value
            return value
        except ChoicesError as exc:
            if not placeholders:
                raise ChoicesError(f"{exp_id}: {exc}") from None
            value = FAST_PLACEHOLDERS[key]
            used[label] = value
            notes.append(f"--fast: no choice for {label}, using the placeholder {value!r}")
            return value

    if exp_id in FIXED_ALIASES or exp_id in BACKBONE_ALIASES:
        target = FIXED_ALIASES[exp_id] if exp_id in FIXED_ALIASES else BACKBONE_ALIASES[exp_id][pick("backbone")]
        inner = resolve_experiment(target, choices, lr=lr, placeholders=placeholders)
        note = f"{exp_id} is an alias of {inner.config.exp_id} (identical training): same run folder, no retraining"
        return ResolvedExperiment(inner.config, exp_id, {**inner.choices_used, **used}, (note, *inner.notes, *notes))

    if base.alias_of_winner:
        mode = pick("input_mode")
        inner = resolve_experiment(f"{base.alias_of_winner}-{mode}", choices, lr=lr, placeholders=placeholders)
        note = f"{exp_id} is an alias of {inner.config.exp_id} (the winning input mode): same run folder, no retraining"
        return ResolvedExperiment(inner.config, exp_id, {**inner.choices_used, **used}, (note, *inner.notes, *notes))

    backbone = base.backbone if base.backbone is not None else pick("backbone")
    mode = base.input_mode if base.input_mode is not None else pick("input_mode")
    if lr is None:
        lr = base.lr if base.lr is not None else pick("lr", backbone)
    config = replace(base, backbone=backbone, input_mode=mode, lr=lr)
    return ResolvedExperiment(config, exp_id, used, tuple(notes))

In [ ]:
#| export
def cmd_download_weights(args: argparse.Namespace) -> int:
    try:
        download_weights(Path(args.data_root))
    except (ExperimentError, OSError) as exc:
        print(f"download-weights: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK

## `run`, `choose-lr`, `predict`, `download-weights` (phase 03)

`run --exp ID` trains and evaluates one experiment (or a group: `E0`, `E1`, `E2`, `X-domain`, or a
comma-separated list) on the given splits and writes `_work/runs/<ID>/split_<k>/` (see `20_classifier.ipynb`).
It is idempotent: finished splits are skipped unless `--force` is given.

**Where results go.** Only a plain `run` writes to `_work/runs/<ID>/`, the folder that `summarize` reads.
Anything that is not the real recipe goes elsewhere and can never be mistaken for a result:

| Flag | Output folder |
|---|---|
| `--sweep-lr` (split 0 only, lr 1e-4 and 3e-4) | `_work/runs/sweep/<ID>_lr<lr>/split_0/` |
| `--fast`, `--lr`, `--epochs`, `--stage1-epochs` (dev overrides) | `_work/runs/scratch/<ID>_fast/` or `<ID>_dev/` (gitignored) |
| `--out-dir DIR` | `DIR/<ID>/` (replaces `_work/runs`) |

`--fast` runs the experiment at smoke scale (24 images per split part, 1 epoch per stage, input size divided by
4, batch 8, real weights only if already downloaded). When `choices.json` does not exist yet it substitutes
placeholder choices and says so, so every experiment ID can be sanity-checked locally in about a minute.
`--batch-size 16 --grad-accum 2` is the manual form of the out-of-memory fallback that `run` applies by itself.

`choose-lr --backbone NAME` implements the T3.4 rule: it reads the two split-0 sweep runs of that backbone,
picks the lr with the higher **validation mixed macro-F1** (ties go to the smaller lr) and records it in
`choices.json` under `lr[NAME]`. It refuses `--fast` and dev-override runs.

**Sweep reuse.** The sweep run at the winning lr is the same training as split 0 of the real run (same config
fingerprint, same split). So `choose-lr` copies it to `_work/runs/<run folder>/split_0/` (plus a `provenance.json`
that says where it came from) and the real `run` then skips split 0. The sweep copy stays in `sweep/` as the evidence
trail. The copy is refused (with the reason printed) when the fingerprint or split sha256 differs, and a split 0 that
already exists in the real folder is never touched.

In [ ]:
#| export
FAST_CONFIG = {"stage1_epochs": 1, "epochs": 1, "size_div": 4, "batch_size": 8}
FAST_IMAGES = 24


def pretrained_available(backbone: str, data_root) -> bool:
    """True if the pretrained file is already stored under <data_root>/_work/weights (no download is triggered)."""
    previous = os.environ.get("TORCH_HOME")
    try:
        set_torch_home(data_root, create=False)
        return weight_file_ok(weight_checkpoint_path(backbone_weights(backbone)))
    finally:
        if previous is None:
            os.environ.pop("TORCH_HOME", None)
        else:
            os.environ["TORCH_HOME"] = previous


def run_destination(data_root, args, cfg: ExperimentConfig, dev: bool) -> tuple[Path, str]:
    """(root folder, run name) for one resolved config; see the table in the notebook."""
    if args.out_dir:
        root = Path(args.out_dir)
    elif dev:
        root = scratch_dir(data_root)
    else:
        root = sweep_dir(data_root) if args.sweep_lr else runs_dir(data_root)
    if args.sweep_lr:
        return root, sweep_run_name(cfg.exp_id, cfg.lr)
    suffix = "" if args.out_dir or not dev else "_fast" if args.fast else "_dev"
    return root, cfg.exp_id + suffix


def apply_run_flags(args, resolved: ResolvedExperiment, data_root) -> tuple[ResolvedExperiment, dict, int | None]:
    """Apply --fast and the dev/system overrides to a resolved experiment. Returns (resolved, overrides, limit)."""
    changes: dict = {}
    overrides: dict = {}
    limit = None
    if args.fast:
        changes.update(FAST_CONFIG)
        changes["pretrained"] = pretrained_available(resolved.config.backbone, data_root)
        overrides["fast"] = True
        limit = FAST_IMAGES
    for flag, field_name in (("epochs", "epochs"), ("stage1_epochs", "stage1_epochs"), ("lr", "lr")):
        value = getattr(args, flag, None)
        if value is not None:
            changes[field_name] = value
            overrides[flag] = value
    for flag in ("batch_size", "grad_accum"):
        value = getattr(args, flag, None)
        if value is not None:
            changes[flag] = value
    notes = resolved.notes
    if args.fast and not changes["pretrained"]:
        notes = (*notes, f"--fast: {resolved.config.backbone} weights are not downloaded, using random initialisation")
    return replace(resolved, config=replace(resolved.config, **changes), notes=notes), overrides, limit


def run_command(args) -> int:
    data_root = Path(args.data_root)
    choices = load_choices(data_root)
    device = resolve_device(args.device)
    ids = expand_experiment_ids(args.exp)
    sweep = args.sweep_lr
    if sweep and args.lr is not None:
        raise ExperimentError("--sweep-lr already sets the learning rates (1e-4 and 3e-4); do not combine it with --lr")
    splits = (0,) if sweep else tuple(args.splits)
    if sweep and tuple(args.splits) != DEFAULT_SPLITS and tuple(args.splits) != (0,):
        print("run: note - --sweep-lr always uses split 0 only (T3.4 rule)")
    dev = bool(args.fast or args.lr is not None or args.epochs is not None or args.stage1_epochs is not None)
    options_base = dict(device=device, seed_offset=args.seed_offset, num_workers=args.num_workers, force=args.force,
                        weights_root=data_root)
    weights_home = set_torch_home(data_root)
    print(f"run: device {device}, data root {data_root}, TORCH_HOME {weights_home}")
    summary = []
    for exp_id in ids:
        for sweep_lr in (SWEEP_LRS if sweep else (None,)):
            resolved = resolve_experiment(exp_id, choices, lr=sweep_lr if sweep else args.lr, placeholders=args.fast)
            resolved, overrides, limit = apply_run_flags(args, resolved, data_root)
            cfg = resolved.config
            if sweep and cfg.exp_id != SWEEP_EXPERIMENT.get(cfg.backbone):
                print(f"run: note - choose-lr reads the sweep of {SWEEP_EXPERIMENT.get(cfg.backbone)} for {cfg.backbone}, not {cfg.exp_id}")
            root, name = run_destination(data_root, args, cfg, dev)
            for note in resolved.notes:
                print(f"run: {note}")
            print(f"run: {resolved.requested} -> {root / name}/split_<k>/" + ("  (sweep, never a result)" if sweep else "  (scratch, never a result)" if dev else ""))
            options = RunOptions(limit=limit, overrides=tuple(overrides.items()), **options_base)
            for split_seed in splits:
                result = run_split(resolved, split_seed, data_root=data_root, out_dir=split_dir(root / name, split_seed),
                                   options=options, log=lambda text: print(text, flush=True))
                metrics = result["metrics"]
                summary.append((name, split_seed, result["status"], metrics["val"]["mixed"]["macro_f1"], metrics["test"]["mixed"]["macro_f1"]))
    print("run: summary (macro-F1)")
    print(f"  {'run':<22}{'split':>5}  {'status':<8}{'val mixed':>10}{'test mixed':>11}")
    for name, split_seed, status, val_f1, test_f1 in summary:
        print(f"  {name:<22}{split_seed:>5}  {status:<8}{fmt3(val_f1):>10}{fmt3(test_f1):>11}")
    return EXIT_OK


def cmd_run(args: argparse.Namespace) -> int:
    try:
        return run_command(args)
    except (ExperimentError, DatasetError) as exc:
        print(f"run: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE


def choose_lr(data_root, backbone: str, out=print) -> dict:
    """T3.4 rule: of the two split-0 sweep runs, take the lr with the higher validation mixed macro-F1."""
    if backbone not in SWEEP_EXPERIMENT:
        raise ExperimentError(f"unknown backbone {backbone!r}, expected one of {', '.join(SWEEP_EXPERIMENT)}")
    exp_id = SWEEP_EXPERIMENT[backbone]
    scores, shas = {}, set()
    for lr in SWEEP_LRS:
        run_dir = split_dir(sweep_dir(data_root) / sweep_run_name(exp_id, lr), 0)
        metrics = read_metrics(run_dir)
        if metrics is None:
            raise ExperimentError(f"missing sweep result {run_dir / 'metrics.json'}; run `run --exp {exp_id} --sweep-lr` first")
        if metrics.get("fast") or metrics.get("overrides"):
            raise ExperimentError(f"{run_dir} is a --fast / dev-override run, not a real sweep run; re-run the sweep without those flags")
        if metrics.get("backbone") != backbone or abs(metrics.get("lr", -1) - lr) > 1e-12:
            raise ExperimentError(f"{run_dir}: expected backbone {backbone} at lr {lr}, found {metrics.get('backbone')} at {metrics.get('lr')}")
        scores[lr] = metrics["val"]["mixed"]["macro_f1"]
        shas.add(metrics["split_sha256"])
    if len(shas) != 1:
        raise ExperimentError("the two sweep runs used different splits; re-run both")
    best = max(SWEEP_LRS, key=lambda lr: (scores[lr], -lr))  # ties: the smaller lr
    evidence = {
        "experiment": exp_id, "split": 0, "split_sha256": shas.pop(),
        "val_mixed_macro_f1": {lr_tag(lr): scores[lr] for lr in SWEEP_LRS},
        "rule": "higher validation mixed macro-F1 on split 0; ties go to the smaller lr",
    }
    path = write_choice(data_root, "lr", best, evidence, sub=backbone)
    out(f"choose-lr: {backbone}: " + ", ".join(f"lr {lr_tag(lr)} -> val mixed F1 {fmt3(scores[lr])}" for lr in SWEEP_LRS)
        + f"  =>  lr {lr_tag(best)}  (written to {path})")
    try:
        reuse = reuse_sweep_run(data_root, backbone, best)
        message = {
            "copied": f"the sweep run at lr {lr_tag(best)} was copied to split 0 of {run_folder_name(exp_id, load_choices(data_root))} "
            "(same training; the real run will skip split 0)",
            "exists": "the real split 0 already exists and was left untouched",
        }[reuse]
    except (ExperimentError, DatasetError) as exc:
        reuse, message = "not reused", f"sweep run not reused for split 0 ({exc}); the real run will train it"
    out(f"choose-lr: {message}")
    return {"lr": best, "evidence": evidence, "split_0": reuse}


PROVENANCE_FILENAME = "provenance.json"


def reuse_sweep_run(data_root, backbone: str, lr: float) -> str:
    """Copy the sweep run at ``lr`` to split 0 of the real run folder of the backbone's sweep experiment.

    Returns "copied" or "exists" (a real split 0 is already there and is left untouched). Raises ExperimentError
    when the sweep run is not the same training as the real run: different config fingerprint (including a changed
    lr in choices.json), different split sha256, or a --fast / dev-override run."""
    data_root = Path(data_root)
    exp_id = SWEEP_EXPERIMENT[backbone]
    choices = load_choices(data_root)
    sweep_split = split_dir(sweep_dir(data_root) / sweep_run_name(exp_id, lr), 0)
    metrics = read_metrics(sweep_split)
    if metrics is None:
        raise ExperimentError(f"no sweep result at {sweep_split / 'metrics.json'}")
    if metrics.get("fast") or metrics.get("overrides"):
        raise ExperimentError(f"{sweep_split} is a --fast / dev-override run")
    resolved = resolve_experiment(exp_id, choices)  # the config a real run would use, with the lr now in choices.json
    expected = config_fingerprint(resolved.config, {"seed_offset": 0, "limit": None})
    if metrics.get("config_fingerprint") != expected:
        raise ExperimentError(
            f"the sweep run's config fingerprint differs from the real {resolved.config.exp_id} run "
            f"(lr in choices.json is {resolved.config.lr:g}, the sweep run used {metrics.get('lr')}, or the recipe changed)"
        )
    if metrics.get("split_sha256") != load_split(data_root, 0)["sha256"]:
        raise ExperimentError("the sweep run's split sha256 differs from the committed split_0")
    real_root = runs_dir(data_root) / run_folder_name(exp_id, choices)
    destination = split_dir(real_root, 0)
    if destination.exists() and any(destination.iterdir()):
        return "exists"
    provenance = {
        "copied_from": "/".join(sweep_split.relative_to(runs_dir(data_root)).parts),
        "reason": "choose-lr: the sweep run at the chosen lr is the same training as split 0 of this run "
        "(identical config fingerprint and split sha256), so it was copied instead of retrained",
        "config_fingerprint": expected, "split_sha256": metrics["split_sha256"], "lr": lr,
    }
    staging = destination.with_name(destination.name + f".tmp{os.getpid()}")
    if staging.exists():
        shutil.rmtree(staging)
    shutil.copytree(sweep_split, staging)
    write_json(staging / PROVENANCE_FILENAME, provenance, indent=2)
    if destination.exists():  # empty leftover folder
        destination.rmdir()
    os.replace(staging, destination)
    return "copied"


def cmd_choose_lr(args: argparse.Namespace) -> int:
    try:
        choose_lr(Path(args.data_root), args.backbone)
    except (ExperimentError, DatasetError) as exc:
        print(f"choose-lr: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK

In [ ]:
#| export
def cmd_predict(args: argparse.Namespace) -> int:
    data_root = Path(args.data_root)
    try:
        if args.run_dir is not None:
            run_dir = Path(args.run_dir)
        else:
            choices = load_choices(data_root)
            exp_id = args.exp or choices.raw("recommended")
            if not exp_id:
                raise ExperimentError(
                    f"no --exp given and {choices.path} has no 'recommended' experiment (written by gate G2, phase 06). "
                    "Pass --exp ID, for example --exp E1-U, or --run-dir PATH"
                )
            run_dir = split_dir(runs_dir(data_root) / run_folder_name(exp_id, choices), args.split)
        if not Path(args.image).is_file():
            raise ExperimentError(f"image {args.image} does not exist")
        predict_image(run_dir, args.image, resolve_device(args.device), data_root)
    except (ExperimentError, DatasetError) as exc:
        print(f"predict: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK

## `gate` and `summarize` (phase 04 / 07)

`gate g0` and `gate g1` recompute a decision of phase 04 from the run folders and write it to `choices.json`
(`input_mode`, resp. `backbone`) together with the evidence; see `50_analysis.ipynb`. `summarize` regenerates
`_work/results/` (tables and `findings.md`) from the run folders, skipping incomplete experiments with a warning.

In [ ]:
#| export
def cmd_gate(args: argparse.Namespace) -> int:
    try:
        run_gate(Path(args.data_root), args.gate)
    except (ExperimentError, DatasetError) as exc:
        print(f"gate {args.gate}: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK


def cmd_summarize(args: argparse.Namespace) -> int:
    try:
        summarize(Path(args.data_root))
    except (ExperimentError, DatasetError) as exc:
        print(f"summarize: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK

In [ ]:
#| export
def resolve_device(requested: str = "auto") -> str:
    """Map the --device flag to a concrete torch device name; raise if it is unavailable."""
    import torch

    available = {
        "cuda": torch.cuda.is_available(),
        "mps": torch.backends.mps.is_available(),
        "cpu": True,
    }
    if requested == "auto":
        return next(name for name in ("cuda", "mps", "cpu") if available[name])
    if not available.get(requested, False):
        raise RuntimeError(f"device {requested!r} was requested but is not available here")
    return requested

In [ ]:
#| export
def cmd_prepare(args: argparse.Namespace) -> int:
    """Index, cache, near-duplicate audit, splits, tables and figures (phase 02), then the white pseudo-masks and QC sheets
    (T5.1) and the annotation selection (T6.1).
    Idempotent."""
    try:
        data_root = Path(args.data_root)
        prepare_data(data_root, workers=args.workers)
        records = load_index(data_root)
        prepare_pseudo_masks(data_root, records, workers=args.workers)
        prepare_annotations(data_root, records)
    except DatasetError as exc:
        print(f"prepare: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    return EXIT_OK


def smoke_split(selected: list[Record], seed: int = 0) -> dict:
    """A deterministic tiny split of the smoke images: every class x domain stratum contributes to train, and
    val / test get the other members alternately (the real splits need >= 7 images per stratum, the smoke
    images have ~2)."""
    parts: dict[str, list[str]] = {"train": [], "val": [], "test": []}
    by_stratum: dict = {}
    for record in sorted(selected, key=lambda r: r.filename):
        by_stratum.setdefault(record.stratum, []).append(record.filename)
    for (cls, domain), names in by_stratum.items():
        even = (CLASS_NAMES.index(cls) + DOMAINS.index(domain)) % 2 == 0
        cycle = ("train", "val" if even else "test", "test" if even else "val", "train")
        for k, name in enumerate(names):
            parts[cycle[k % 4]].append(name)
    train, val, test = (sorted(parts[key]) for key in ("train", "val", "test"))
    return {"seed": seed, "train": train, "val": val, "test": test, "sha256": split_sha256(seed, train, val, test)}


def classifier_smoke(data_root, tmp_root, selected: list[Record], log=print) -> None:
    """One tiny experiment (E0-rect, ResNet50, 1 epoch per stage, input size / 4) through the real `run` code
    path on CPU, with a complete run folder written under ``tmp_root``; then `predict` on one image of it."""
    started = time.time()
    write_index(tmp_root, selected)
    split = smoke_split(selected)
    smoke_require(all(split[k] for k in ("train", "val", "test")), "tiny split has an empty part")
    write_json(split_path(tmp_root, 0), split)
    resolved = resolve_experiment("E0-rect", Choices(choices_path(tmp_root), {}), lr=3e-4)
    pretrained = pretrained_available(resolved.config.backbone, data_root)
    resolved = replace(resolved, config=replace(resolved.config, pretrained=pretrained, **FAST_CONFIG))
    options = RunOptions(device="cpu", num_workers=2, weights_root=Path(data_root))
    run_dir = split_dir(tmp_root / WORK_DIRNAME / "runs" / "E0-rect_smoke", 0)
    lines: list[str] = []
    result = run_split(resolved, 0, data_root=tmp_root, out_dir=run_dir, options=options, log=lines.append)
    smoke_require(result["status"] == "done", "the tiny run did not train")
    for name in (*RUN_FILES, "best.pt"):
        smoke_require((run_dir / name).is_file(), f"run folder lacks {name}")
    smoke_require(not list(run_dir.glob("*.tmp*")), "temporary files left behind (non-atomic write)")
    metrics = read_metrics(run_dir)
    smoke_require(metrics["split_sha256"] == split["sha256"] and metrics["wall_time_s"] > 0, "metrics.json is incomplete")
    for part in ("val", "test"):
        with open(run_dir / f"predictions_{part}.csv", newline="") as handle:
            rows = list(csv.reader(handle))
        smoke_require(rows[0][:4] == ["filename", "domain", "true", "pred"] and len(rows) - 1 == len(split[part]),
                      f"predictions_{part}.csv has the wrong rows")
    again = run_split(resolved, 0, data_root=tmp_root, out_dir=run_dir, options=options, log=lines.append)
    smoke_require(again["status"] == "skipped", "a finished split was not skipped (idempotence)")
    image = Path(data_root) / selected[0].rel_path
    probs = predict_image(run_dir, image, "cpu", data_root, out=lambda text: None)
    smoke_require(probs.shape == (NUM_CLASSES,) and abs(float(probs.sum()) - 1.0) < 1e-4, "predict returned bad probabilities")
    log(f"smoke: OK - tiny {resolved.config.exp_id} run ({'pretrained' if pretrained else 'random'} weights, "
        f"{len(split['train'])}/{len(split['val'])}/{len(split['test'])} train/val/test images, workers 2): complete run folder, "
        f"skipped on rerun, predict OK ({time.time() - started:.0f} s)")


def cmd_smoke(args: argparse.Namespace) -> int:
    """Tiny end-to-end check: data root, image discovery, library imports, device, the data path (index, EXIF
    read, cache, dHash, splits, figures on ~20 images in a temp folder) and one tiny classifier experiment
    through the real `run` code path on CPU, followed by `predict` (everything in a temp folder)."""
    data_root = Path(args.data_root)
    print(f"smoke: data root = {data_root.resolve()}")
    if not data_root.is_dir():
        print(f"smoke: FAIL - data root {data_root} is not a directory (see --data-root)", file=sys.stderr)
        return EXIT_FAILURE
    images = find_images(data_root)
    if not images:
        print(f"smoke: FAIL - no JPEG images found under {data_root}", file=sys.stderr)
        return EXIT_FAILURE
    print(f"smoke: OK - found {len(images)} JPEG images (excluding {WORK_DIRNAME}/)")

    import cv2
    import matplotlib
    import numpy
    import skimage
    import sklearn
    import torch
    import torchvision

    print(f"smoke: python {sys.version.split()[0]}")
    for module in (numpy, matplotlib, cv2, sklearn, skimage, torch, torchvision):
        print(f"smoke: {module.__name__} {module.__version__}")
    device = resolve_device(args.device)
    print(f"smoke: OK - device = {device} (requested: {args.device}), fast = {args.fast}")
    try:
        run_data_smoke(data_root, extra_checks=lambda tmp_root, selected: classifier_smoke(data_root, tmp_root, selected))
    except (DatasetError, ExperimentError) as exc:
        print(f"smoke: FAIL - {exc}", file=sys.stderr)
        return EXIT_FAILURE
    print("smoke: PASSED")
    return EXIT_OK


def not_implemented(command: str, phase: str) -> Callable[[argparse.Namespace], int]:
    """Placeholder handler: say which phase will provide the command and exit non-zero."""

    def handler(args: argparse.Namespace) -> int:
        print(f"{command}: not implemented yet (phase {phase})", file=sys.stderr)
        return EXIT_NOT_IMPLEMENTED

    return handler

In [ ]:
#| export
def parse_splits(text: str) -> tuple[int, ...]:
    """Parse '0,1,2' into (0, 1, 2); duplicates are dropped, order is kept."""
    try:
        values = [int(part) for part in text.split(",") if part.strip() != ""]
    except ValueError:
        raise argparse.ArgumentTypeError(f"expected comma-separated integers, got {text!r}") from None
    if not values or min(values) < 0:
        raise argparse.ArgumentTypeError(f"expected one or more non-negative integers, got {text!r}")
    return tuple(dict.fromkeys(values))


def add_global_flags(parser: argparse.ArgumentParser, *, is_subcommand: bool) -> None:
    """Add the global flags. On subparsers the defaults are suppressed so that a flag given
    before the command is not overwritten by the subparser's default."""

    def default(value):
        return argparse.SUPPRESS if is_subcommand else value

    group = parser.add_argument_group("global flags (accepted before or after the command)")
    group.add_argument(
        "--data-root",
        type=Path,
        default=default(Path(DEFAULT_DATA_ROOT)),
        help=f"dataset folder, relative to the working directory (default: {DEFAULT_DATA_ROOT})",
    )
    group.add_argument(
        "--device",
        choices=DEVICE_CHOICES,
        default=default("auto"),
        help="compute device (default: auto = cuda, then mps, then cpu)",
    )
    group.add_argument(
        "--seed-offset",
        type=int,
        default=default(0),
        help="added to every seed, for repeating an experiment with different randomness (default: 0)",
    )
    group.add_argument(
        "--fast",
        action="store_true",
        default=default(False),
        help="smoke settings: tiny subsets, 1 epoch",
    )


def _configure_prepare(parser: argparse.ArgumentParser) -> None:
    parser.add_argument(
        "--workers",
        type=int,
        default=None,
        metavar="N",
        help="worker processes for image caching (default: number of CPU cores, at most 8)",
    )


def _configure_run(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--exp", required=True, metavar="ID", help="experiment ID (E1-U), a group (E0, E1, E2, X-domain) or a comma-separated list")
    parser.add_argument(
        "--splits",
        type=parse_splits,
        default=DEFAULT_SPLITS,
        metavar="LIST",
        help="comma-separated split seeds (default: 0,1,2,3,4)",
    )
    parser.add_argument("--force", action="store_true", help="retrain splits whose metrics.json already exists")
    parser.add_argument("--sweep-lr", action="store_true",
                        help="learning-rate sweep: split 0 at lr 1e-4 and 3e-4 into _work/runs/sweep/ (never a result)")
    parser.add_argument("--batch-size", type=int, default=None, metavar="N", help="batch size (default 32; 16 + --grad-accum 2 if memory is short)")
    parser.add_argument("--grad-accum", type=int, default=None, metavar="N", help="gradient accumulation steps (default 1)")
    parser.add_argument("--num-workers", type=int, default=None, metavar="N", help="data-loading processes (default: cores - 1, at most 4)")
    parser.add_argument("--out-dir", type=Path, default=None, metavar="DIR", help="write DIR/<ID>/split_<k>/ instead of _work/runs/<ID>/")
    dev = parser.add_argument_group("dev overrides (output goes to _work/runs/scratch/, never counted as a result)")
    dev.add_argument("--lr", type=float, default=None, metavar="LR", help="fine-tuning learning rate instead of choices.json")
    dev.add_argument("--epochs", type=int, default=None, metavar="N", help="fine-tuning epochs (default 30)")
    dev.add_argument("--stage1-epochs", type=int, default=None, metavar="N", help="linear-probe epochs (default 2)")


def _configure_choose_lr(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--backbone", required=True, choices=BACKBONES, help="backbone whose split-0 sweep to evaluate")


def _configure_gate(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("gate", choices=("g0", "g1"), help="g0: input mode (after E0), g1: backbone (after E1-U, E2-cnx-U, E2-swin-U)")


def _configure_gradcam(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--exp", required=True, metavar="ID", help="experiment ID")
    parser.add_argument("--split", type=int, required=True, metavar="K", help="split seed")


def _configure_predict(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--exp", default=None, metavar="ID", help="experiment ID (default: the recommended one in choices.json)")
    parser.add_argument("--split", type=int, default=0, metavar="K", help="split seed (default: 0)")
    parser.add_argument("--image", type=Path, required=True, metavar="PATH", help="image to classify")
    parser.add_argument("--run-dir", type=Path, default=None, metavar="DIR", help="use DIR/best.pt directly instead of the run folder of --exp")


@dataclass(frozen=True)
class Command:
    name: str
    help: str
    handler: Callable[[argparse.Namespace], int]
    configure: Callable[[argparse.ArgumentParser], None] | None = None


COMMANDS = (
    Command(
        "download-weights",
        "fetch the torchvision pretrained weights into <data-root>/_work/weights/ (idempotent)",
        cmd_download_weights,
    ),
    Command(
        "prepare",
        "index, cache, near-duplicate audit, splits, tables and figures (idempotent)",
        cmd_prepare,
        _configure_prepare,
    ),
    Command("run", "train and evaluate one experiment over the given splits", cmd_run, _configure_run),
    Command(
        "choose-lr",
        "record the learning rate of a backbone from its split-0 sweep (T3.4 rule) in choices.json",
        cmd_choose_lr,
        _configure_choose_lr,
    ),
    Command(
        "gate",
        "recompute decision gate g0 (input mode) or g1 (backbone) from the runs and record it in choices.json",
        cmd_gate,
        _configure_gate,
    ),
    Command("summarize", "aggregate all runs into tables and findings (figures follow in phase 07)", cmd_summarize),
    Command("gradcam", "Grad-CAM figures for one experiment and split", not_implemented("gradcam", "07"), _configure_gradcam),
    Command("predict", "classify a single image and print class probabilities", cmd_predict, _configure_predict),
    Command("smoke", "tiny end-to-end check: imports, device, data path, one tiny experiment and predict", cmd_smoke),
    Command(
        "reproduce",
        "download-weights, prepare, every reported experiment, summarize",
        not_implemented("reproduce", "08"),
    ),
)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog=Path(__file__).name,
        description="CSCI935 rice leaf disease classification - reproduces every reported number.",
        epilog="Global flags may be placed before or after the command.",
    )
    add_global_flags(parser, is_subcommand=False)
    subparsers = parser.add_subparsers(dest="command", required=True, metavar="COMMAND")
    for command in COMMANDS:
        sub = subparsers.add_parser(command.name, help=command.help, description=command.help)
        if command.configure is not None:
            command.configure(sub)
        add_global_flags(sub, is_subcommand=True)
        sub.set_defaults(handler=command.handler)
    return parser

In [ ]:
#| export
def run_cli(argv: list[str] | None = None) -> int:
    """Parse argv and run the chosen command. Returns the process exit code."""
    args = build_parser().parse_args(argv)
    return args.handler(args)


def main() -> None:
    sys.exit(run_cli())